# Ingenieria de variables

> **Entrada:** `df` = `base_com` (388K filas, granularidad transacción)  
> **salida:** `df_model` (1 fila por cliente, lista para modelado)

In [ ]:
import pandas as pd
import numpy as np
from google.colab import files

In [ ]:
#Abrir base_com
uploaded = files.upload()
file_name = list(uploaded.keys())[0]
df = pd.read_csv(file_name)
print(df.shape)
df.head()

Saving base_com.csv to base_com.csv
(388048, 24)


,customerID,customerType,riskLevel,investmentCapacity,lastQuestionnaireDate,timestamp_x,ISIN,transactionID,transactionType,timestamp_y,...,assetName,assetShortName,assetCategory,assetSubCategory,asset_marketID,sector,industry,profitability,country,marketClass
0,00017496858921195E5A,Professional,Aggressive,CAP_GT300K,2020-03-13,2021-03-19,GRS434003000,7590224,Buy,2020-03-27,...,PPC SA,ΔΕΗ,Stock,Large Cap,XATH,Utilities,Utilities - Renewable,2.211538,Greece,Public Securities
1,00017496858921195E5A,Professional,Aggressive,CAP_GT300K,2020-03-13,2021-03-19,GRS434003000,7607029,Sell,2020-04-06,...,PPC SA,ΔΕΗ,Stock,Large Cap,XATH,Utilities,Utilities - Renewable,2.211538,Greece,Public Securities
2,00017496858921195E5A,Professional,Aggressive,CAP_GT300K,2020-03-13,2021-03-19,GRS434003000,7634872,Buy,2020-04-24,...,PPC SA,ΔΕΗ,Stock,Large Cap,XATH,Utilities,Utilities - Renewable,2.211538,Greece,Public Securities
3,00017496858921195E5A,Professional,Aggressive,CAP_GT300K,2020-03-13,2021-03-19,GRS434003000,7652627,Sell,2020-05-07,...,PPC SA,ΔΕΗ,Stock,Large Cap,XATH,Utilities,Utilities - Renewable,2.211538,Greece,Public Securities
4,00017496858921195E5A,Professional,Aggressive,CAP_GT300K,2020-03-13,2021-03-19,GRS434003000,7664807,Buy,2020-05-15,...,PPC SA,ΔΕΗ,Stock,Large Cap,XATH,Utilities,Utilities - Renewable,2.211538,Greece,Public Securities


## 1. Separar compras y ventas
Toda la lógica de features descansa en esta separación.  
Normalizar a minúsculas evita errores silenciosos si el dato viene como `BUY`, `Buy` o `buy`.

In [ ]:
# Normalizar y separar
_type = df['transactionType'].str.strip().str.lower()
df_buy  = df[_type == 'buy'].copy()
df_sell = df[_type == 'sell'].copy()

print(f"Transacciones BUY  : {len(df_buy):>10,}")
print(f"Transacciones SELL : {len(df_sell):>10,}")
print(f"Total              : {len(df):>10,}")
print(f"Cobertura          : {(len(df_buy)+len(df_sell))/len(df)*100:.2f}%")

Transacciones BUY  :    228,913
Transacciones SELL :    159,135
Total              :    388,048
Cobertura          : 100.00%


## 2. Agregaciones de compras (`agg_buy`)
Cada métrica captura una dimensión distinta del comportamiento comprador:
- **Conteo/volumen** → intensidad de actividad
- **Valor** → tamaño de las apuestas
- **Profitability** → calidad de los activos elegidos al comprar

In [ ]:
agg_buy = (
    df_buy.groupby('customerID')
    .agg(
        # Actividad
        n_buy              = ('transactionID',  'count'),
        # Volumen en unidades
        total_units_buy    = ('units',          'sum'),
        avg_units_buy      = ('units',          'mean'),
        # Volumen en valor monetario
        total_value_buy    = ('totalValue',     'sum'),
        avg_value_buy      = ('totalValue',     'mean'),
        # Calidad de activos comprados
        avg_profit_buy     = ('profitability',  'mean'),
        profit_std_buy     = ('profitability',  'std'),
        # Diversificación (se recalculará globalmente más abajo)
        n_assets_buy       = ('ISIN',           'nunique'),
        n_categories_buy   = ('assetCategory',  'nunique'),
        n_subcategories_buy= ('assetSubCategory','nunique'),
    )
)
print(f"Clientes con al menos 1 compra: {len(agg_buy):,}")

Clientes con al menos 1 compra: 29,090


## 3. Agregaciones de ventas (`agg_sell`)

In [ ]:
agg_sell = (
    df_sell.groupby('customerID')
    .agg(
        n_sell              = ('transactionID',   'count'),
        total_units_sell    = ('units',           'sum'),
        avg_units_sell      = ('units',           'mean'),
        total_value_sell    = ('totalValue',      'sum'),
        avg_value_sell      = ('totalValue',      'mean'),
        avg_profit_sell     = ('profitability',   'mean'),
        profit_std_sell     = ('profitability',   'std'),
        n_assets_sell       = ('ISIN',            'nunique'),
        n_categories_sell   = ('assetCategory',   'nunique'),
        n_subcategories_sell= ('assetSubCategory','nunique'),
    )
)
print(f"Clientes con al menos 1 venta: {len(agg_sell):,}")

Clientes con al menos 1 venta: 22,332


## 4. Combinar con `fillna` selectivo


In [ ]:
# JOIN incluye clientes que solo compran o solo venden
customer_features = (
    agg_buy
    .join(agg_sell, how='outer')
    .reset_index()
)

count_sum_cols = [
    'n_buy', 'total_units_buy', 'avg_units_buy',
    'total_value_buy', 'avg_value_buy',
    'n_assets_buy', 'n_categories_buy', 'n_subcategories_buy',
    'n_sell', 'total_units_sell', 'avg_units_sell',
    'total_value_sell', 'avg_value_sell',
    'n_assets_sell', 'n_categories_sell', 'n_subcategories_sell',
]
customer_features[count_sum_cols] = customer_features[count_sum_cols].fillna(0)


print(f"Clientes totales en customer_features: {len(customer_features):,}")
print(f"NaN en avg_profit_sell (clientes solo-buy): {customer_features['avg_profit_sell'].isna().sum():,}")

Clientes totales en customer_features: 29,090
NaN en avg_profit_sell (clientes solo-buy): 6,758


## 5. Diversificación


In [ ]:
# Diversificación sobre TODA la actividad (buy + sell combinados)
div_global = (
    df.groupby('customerID')
    .agg(
        asset_diversification       = ('ISIN',             'nunique'),
        category_diversification    = ('assetCategory',    'nunique'),
        subcategory_diversification = ('assetSubCategory', 'nunique'),
    )
    .reset_index()
)

# Reemplazar las columnas de diversificación infladas
customer_features = customer_features.merge(div_global, on='customerID', how='left')

print("Ejemplo de diversificación:")
print(customer_features[['customerID','asset_diversification',
                           'category_diversification',
                           'subcategory_diversification']].head())

Ejemplo de diversificación:
             customerID  asset_diversification  category_diversification  \
0  00017496858921195E5A                     13                         1   
1  00024864C985E72167A0                      1                         1   
2  0004718496C71D464F57                     14                         2   
3  000676D07A4CF7526ECB                      1                         1   
4  000900E880281981624D                      1                         1   

   subcategory_diversification  
0                            1  
1                            1  
2                            2  
3                            1  
4                            1  


## 6. Features temporales
Capturan **cuándo** y **con qué frecuencia** opera el cliente:
- `activity_days`: ventana temporal total — distingue traders activos de ocasionales
- `recency_days`: días desde la última operación — clientes inactivos vs. activos hoy
- `avg_days_between_ops`: cadencia promedio — detecta patrones sistemáticos vs. espontáneos

In [ ]:
# Fecha de corte: última fecha observable en los datos
cutoff_date = pd.to_datetime(df['timestamp_y']).max()

time_features = (
    df.groupby('customerID')['timestamp_y']
    .agg(
        first_transaction = 'min',
        last_transaction  = 'max',
        n_op_dates        = 'nunique',   # días distintos con actividad
    )
    .reset_index()
)

time_features['first_transaction'] = pd.to_datetime(time_features['first_transaction'])
time_features['last_transaction']  = pd.to_datetime(time_features['last_transaction'])

# Ventana temporal total en días
time_features['activity_days'] = (
    time_features['last_transaction'] - time_features['first_transaction']
).dt.days

# Recencia: días desde la última operación hasta el corte
time_features['recency_days'] = (
    cutoff_date - time_features['last_transaction']
).dt.days

# Cadencia promedio entre operaciones
# activity_days / (n_op_dates - 1) — evitar división por 0
time_features['avg_days_between_ops'] = np.where(
    time_features['n_op_dates'] > 1,
    time_features['activity_days'] / (time_features['n_op_dates'] - 1),
    np.nan   # cliente con 1 sola fecha: no hay cadencia calculable
)

# Unir al dataset principal
customer_features = customer_features.merge(
    time_features[['customerID','activity_days','recency_days','avg_days_between_ops']],
    on='customerID', how='left'
)

print(f"activity_days   — media: {customer_features['activity_days'].mean():.0f} días")
print(f"recency_days    — media: {customer_features['recency_days'].mean():.0f} días")
print(f"avg_days_bet_op — media: {customer_features['avg_days_between_ops'].mean():.1f} días")

activity_days   — media: 890 días
recency_days    — media: 530 días
avg_days_bet_op — media: 624.3 días


## 7. Indicadores derivados
Cada indicador captura una **hipótesis de comportamiento** específica:

| Feature | Hipótesis |
|---|---|
| `n_transactions` | Volumen total de actividad |
| `buy_sell_ratio` | ¿Acumula o rota? >1 acumulador, <1 vendedor neto |
| `net_units` | Posición neta en unidades |
| `net_value` | Posición neta en valor monetario |
| `risk_exposure` | Volumen bruto total operado (tamaño del inversor) |
| `profitability_diff` | ¿Compra mejor de lo que vende? |


In [ ]:
# Actividad total
customer_features['n_transactions'] = (
    customer_features['n_buy'] + customer_features['n_sell']
)

# Ratio compra/venta — +1 en denominador evita división por cero
customer_features['buy_sell_ratio'] = (
    customer_features['n_buy'] / (customer_features['n_sell'] + 1)
)

customer_features['net_units'] = (
    customer_features['total_units_buy'] - customer_features['total_units_sell']
)
customer_features['net_value'] = (
    customer_features['total_value_buy'] - customer_features['total_value_sell']
)

customer_features['risk_exposure'] = (
    customer_features['total_value_buy'] + customer_features['total_value_sell']
)

customer_features['profitability_diff'] = (
    customer_features['avg_profit_buy'] - customer_features['avg_profit_sell']
)

print("Features derivados calculados.")
print(customer_features[['customerID','n_transactions','buy_sell_ratio',
                           'net_value','risk_exposure','profitability_diff']].head())

Features derivados calculados.
             customerID  n_transactions  buy_sell_ratio      net_value  \
0  00017496858921195E5A           124.0        1.777778   30633.105000   
1  00024864C985E72167A0             1.0        1.000000    4999.993985   
2  0004718496C71D464F57            36.0        1.846154  283881.745000   
3  000676D07A4CF7526ECB             1.0        1.000000    4962.784731   
4  000900E880281981624D             1.0        1.000000     700.000000   

   risk_exposure  profitability_diff  
0  728451.013000            0.145671  
1    4999.993985                 NaN  
2  430114.335000           -0.073759  
3    4962.784731                 NaN  
4     700.000000                 NaN  


## 8. Incorporar perfil del cliente
Se unen las variables categóricas del cliente (`riskLevel`, `investmentCapacity`, `customerType`).  
Estas son las variables que el modelo usará como **contexto** del comportamiento.

In [ ]:
from google.colab import files

uploaded = files.upload()


file_name = list(uploaded.keys())[0]


cus = pd.read_csv(file_name)


print(cus.shape)
cus.head()

Saving customers_dedup.csv to customers_dedup (1).csv
(29090, 6)


,customerID,customerType,riskLevel,investmentCapacity,lastQuestionnaireDate,timestamp
0,00017496858921195E5A,Professional,Aggressive,CAP_GT300K,2020-03-13,2021-03-19
1,00024864C985E72167A0,Mass,Predicted_Conservative,Predicted_CAP_LT30K,2000-01-01,2021-09-09
2,0004718496C71D464F57,Mass,Predicted_Conservative,Predicted_CAP_80K_300K,2000-01-01,2018-01-02
3,000676D07A4CF7526ECB,Mass,Income,CAP_30K_80K,2020-11-02,2022-11-22
4,000900E880281981624D,Mass,Balanced,CAP_LT30K,2018-07-31,2021-07-16


In [ ]:
# Columnas del perfil del cliente (snapshot más reciente)
cus_small = cus[['customerID', 'riskLevel', 'investmentCapacity', 'customerType']].copy()

df_model = customer_features.merge(cus_small, on='customerID', how='left')

print(f"Shape final de df_model: {df_model.shape}")
print(f"Clientes únicos: {df_model['customerID'].nunique():,}")
print(f"\nColumnas ({df_model.shape[1]}):")
for col in df_model.columns:
    dtype = str(df_model[col].dtype)
    nulls = df_model[col].isna().sum()
    null_pct = nulls / len(df_model) * 100
    flag = ' ⚠️' if null_pct > 5 else ''
    print(f"  {col:35s} {dtype:12s}  nulls: {null_pct:5.1f}%{flag}")

Shape final de df_model: (29090, 36)
Clientes únicos: 29,090

Columnas (36):
  customerID                          object        nulls:   0.0%
  n_buy                               int64         nulls:   0.0%
  total_units_buy                     float64       nulls:   0.0%
  avg_units_buy                       float64       nulls:   0.0%
  total_value_buy                     float64       nulls:   0.0%
  avg_value_buy                       float64       nulls:   0.0%
  avg_profit_buy                      float64       nulls:   0.0%
  profit_std_buy                      float64       nulls:  47.8% ⚠️
  n_assets_buy                        int64         nulls:   0.0%
  n_categories_buy                    int64         nulls:   0.0%
  n_subcategories_buy                 int64         nulls:   0.0%
  n_sell                              float64       nulls:   0.0%
  total_units_sell                    float64       nulls:   0.0%
  avg_units_sell                      float64       nulls:   0

## 10. Exportar

In [ ]:
from google.colab import files

df_model.to_csv("df_model.csv", index=False, encoding='utf-8')
print(f"✅ df_model.csv exportado — {df_model.shape[0]:,} filas × {df_model.shape[1]} columnas")
files.download("df_model.csv")

✅ df_model.csv exportado — 29,090 filas × 36 columnas


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>